# Cached-Feature Image Captioning

This notebook trains a Transformer caption decoder from frozen ResNet-34 spatial features.

Dimension names used throughout:

- `B_img`: unique images in a batch
- `C_img`: cached ResNet channels (`512`)
- `H_img × W_img`: cached spatial grid (`7 × 7`)
- `S_img`: flattened image positions (`49`)
- `B_cap`: captions in a batch (normally `5 × B_img`)
- `T`: padded caption length
- `E`: decoder embedding dimension
- `V`: tokenizer vocabulary size


## 1. Imports


In [38]:
# PURPOSE: Import dependencies used by preprocessing, datasets, and training.
# INPUT SHAPE: None.
# OUTPUT SHAPE: None; modules and helper classes become available.

import csv
import json
import math
import random
import re
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence


## 2. Configuration and GPU setup


In [39]:
# PURPOSE: Define reproducible paths, dimensions, and GPU training settings.
# INPUT SHAPE: Cached features must have shape [N_images, 512, 7, 7].
# OUTPUT SHAPE: Scalar configuration values and CUDA device handle.

SEED = 42
IMAGE_DIR = Path("../datasets/Captioning/Images")
CAPTION_FILE = Path("../datasets/Captioning/captions.txt")
FEATURE_CACHE_PATH = Path("image_feature_cache/resnet34_spatial_fp16.pt")
TOKENIZER_PATH = Path("image_feature_cache/caption_tokenizer.json")
CHECKPOINT_DIR = Path("checkpoints")

TRAIN_SIZE = 0.70
TEST_SIZE = 0.20
VAL_SIZE = 0.10

IMAGE_BATCH_SIZE = 8       # B_img=8 normally produces B_cap=40 captions.
EMBED_DIM = 256            # E
ATTENTION_HEADS = 8
DECODER_LAYERS = 2
DROPOUT = 0.10
MAX_VOCAB_SIZE = 10_000
MIN_TOKEN_FREQUENCY = 2
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-2
EPOCHS = 100

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for this training notebook")

device = torch.device("cuda:0")
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True

print("Training device:", torch.cuda.get_device_name(0))


Training device: NVIDIA GeForce RTX 4060 Laptop GPU


## 3. Fast word-level caption tokenizer


In [41]:
# PURPOSE: Build a compact word vocabulary and convert captions to token IDs.
# INPUT SHAPE: One caption is a Python string; a corpus is list[str].
# OUTPUT SHAPE: encode(text) -> list[int] with length L; decode(ids) -> str.

class CaptionTokenizer:
    SPECIAL_TOKENS = ("<PAD>", "<BOS>", "<EOS>", "<UNK>")
    TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")

    def __init__(self, max_vocab_size=10_000, min_frequency=2):
        if max_vocab_size < len(self.SPECIAL_TOKENS):
            raise ValueError("max_vocab_size is smaller than the special-token set")

        self.max_vocab_size = max_vocab_size
        self.min_frequency = min_frequency
        self.token_to_id = {
            token: index for index, token in enumerate(self.SPECIAL_TOKENS)
        }
        self.id_to_token = list(self.SPECIAL_TOKENS)

    @property
    def pad_id(self):
        return self.token_to_id["<PAD>"]

    @property
    def bos_id(self):
        return self.token_to_id["<BOS>"]

    @property
    def eos_id(self):
        return self.token_to_id["<EOS>"]

    @property
    def unk_id(self):
        return self.token_to_id["<UNK>"]

    @property
    def vocab_size(self):
        return len(self.id_to_token)

    def tokenize(self, text):
        # Input: str. Output: list[str] of length L.
        return self.TOKEN_PATTERN.findall(text.lower())

    def fit(self, captions):
        # Input: iterable[str]. Output: vocabulary with V token IDs.
        token_counts = Counter()
        for caption in captions:
            token_counts.update(self.tokenize(caption))

        capacity = self.max_vocab_size - len(self.SPECIAL_TOKENS)
        vocabulary = [
            token
            for token, frequency in token_counts.most_common()
            if frequency >= self.min_frequency
        ][:capacity]

        for token in vocabulary:
            self.token_to_id[token] = len(self.id_to_token)
            self.id_to_token.append(token)

    def encode(self, text):
        # Input: str. Output: list[int] of length L (without BOS/EOS).
        return [
            self.token_to_id.get(token, self.unk_id)
            for token in self.tokenize(text)
        ]

    def encode_pair(self, text):
        # Input: str. Outputs: input_ids [L+1], target_ids [L+1].
        token_ids = self.encode(text)
        return [self.bos_id, *token_ids], [*token_ids, self.eos_id]

    def decode(self, token_ids, remove_special_tokens=True):
        # Input: iterable[int] of length L. Output: decoded str.
        ignored = {self.pad_id, self.bos_id, self.eos_id}
        tokens = []
        for token_id in token_ids:
            token_id = int(token_id)
            if remove_special_tokens and token_id in ignored:
                continue
            tokens.append(
                self.id_to_token[token_id]
                if 0 <= token_id < self.vocab_size
                else "<UNK>"
            )
        return " ".join(tokens)

    def save(self, path):
        # Input: vocabulary of size V. Output: one JSON tokenizer file.
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(
            json.dumps(
                {
                    "max_vocab_size": self.max_vocab_size,
                    "min_frequency": self.min_frequency,
                    "id_to_token": self.id_to_token,
                },
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    @classmethod
    def load(cls, path):
        # Input: tokenizer JSON. Output: restored CaptionTokenizer with V tokens.
        state = json.loads(Path(path).read_text(encoding="utf-8"))
        tokenizer = cls(state["max_vocab_size"], state["min_frequency"])
        tokenizer.id_to_token = state["id_to_token"]
        tokenizer.token_to_id = {
            token: index for index, token in enumerate(tokenizer.id_to_token)
        }
        return tokenizer


## 4. Cached, image-grouped dataset components


In [ ]:
# PURPOSE: Represent one cached image once and collate all of its captions.
# DATASET INPUT SHAPE: feature [512,7,7] plus caption pairs with variable [L+1].
# COLLATE OUTPUT SHAPES: features [B_img,512,7,7], captions [B_cap,T], owners [B_cap].

class GroupedCachedCaptionDataset(Dataset):
    def __init__(self, image_names, feature_bank, feature_index, encoded_captions):
        self.image_names = list(image_names)
        self.feature_bank = feature_bank
        self.feature_index = feature_index
        self.encoded_captions = encoded_captions

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, index):
        image_name = self.image_names[index]
        feature = self.feature_bank[self.feature_index[image_name]]  # [512,7,7]
        caption_pairs = self.encoded_captions[image_name]
        return feature, caption_pairs


class GroupedCaptionCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        feature_maps = []
        input_captions = []
        target_captions = []
        caption_image_indices = []

        for image_index, (feature, caption_pairs) in enumerate(batch):
            feature_maps.append(feature)                              # [512,7,7]
            for input_ids, target_ids in caption_pairs:
                input_captions.append(input_ids)                      # [L+1]
                target_captions.append(target_ids)                    # [L+1]
                caption_image_indices.append(image_index)

        return (
            torch.stack(feature_maps),                                # [B_img,512,7,7]
            pad_sequence(
                input_captions,
                batch_first=True,
                padding_value=self.pad_id,
            ),                                                        # [B_cap,T]
            pad_sequence(
                target_captions,
                batch_first=True,
                padding_value=self.pad_id,
            ),                                                        # [B_cap,T]
            torch.tensor(caption_image_indices, dtype=torch.long),     # [B_cap]
        )


## 5. Caption data preparation pipeline


In [43]:
# PURPOSE: Encapsulate CSV parsing, splitting, tokenization, cache loading, and loaders.
# INPUT SHAPES: CSV rows [N_captions], feature cache [N_images,512,7,7].
# OUTPUT SHAPE: CaptionDataBundle containing three grouped datasets and DataLoaders.

@dataclass
class CaptionDataBundle:
    tokenizer: CaptionTokenizer
    feature_bank: torch.Tensor
    feature_index: dict
    encoded_captions: dict
    train_image_names: list
    val_image_names: list
    test_image_names: list
    train_dataset: Dataset
    val_dataset: Dataset
    test_dataset: Dataset
    train_loader: DataLoader
    val_loader: DataLoader
    test_loader: DataLoader
    maximum_caption_length: int


class CaptionDataPipeline:
    def __init__(
        self,
        caption_file,
        feature_cache_path,
        tokenizer_path,
        train_size=0.70,
        test_size=0.20,
        val_size=0.10,
        max_vocab_size=10_000,
        min_token_frequency=2,
        image_batch_size=8,
        seed=42,
    ):
        if not np.isclose(train_size + test_size + val_size, 1.0):
            raise ValueError("train_size + test_size + val_size must equal 1")

        self.caption_file = Path(caption_file)
        self.feature_cache_path = Path(feature_cache_path)
        self.tokenizer_path = Path(tokenizer_path)
        self.train_size = train_size
        self.test_size = test_size
        self.val_size = val_size
        self.max_vocab_size = max_vocab_size
        self.min_token_frequency = min_token_frequency
        self.image_batch_size = image_batch_size
        self.seed = seed

    def _read_captions(self):
        # Input: caption CSV with N_captions rows.
        # Output: dict[image_name, list[str]], normally 5 captions per image.
        captions_by_image = {}
        with self.caption_file.open("r", encoding="utf-8", newline="") as file:
            for row in csv.DictReader(file):
                captions_by_image.setdefault(row["image"], []).append(row["caption"])

        if not captions_by_image:
            raise ValueError(f"No captions found in {self.caption_file}")
        return captions_by_image

    def _split_images(self, image_names):
        # Input: list[str] of length N_images.
        # Output: mutually exclusive train/validation/test image-name lists.
        train_names, val_names = train_test_split(
            sorted(image_names),
            test_size=self.val_size,
            shuffle=True,
            random_state=self.seed,
        )
        adjusted_test_size = self.test_size / (
            self.train_size + self.test_size
        )
        train_names, test_names = train_test_split(
            train_names,
            test_size=adjusted_test_size,
            shuffle=True,
            random_state=self.seed,
        )
        return list(train_names), list(val_names), list(test_names)

    def _fit_and_encode(self, captions_by_image, train_image_names):
        # Input: training caption strings.
        # Output: tokenizer of size V and tensors [L+1] for every caption.
        tokenizer = CaptionTokenizer(
            max_vocab_size=self.max_vocab_size,
            min_frequency=self.min_token_frequency,
        )
        tokenizer.fit(
            caption
            for image_name in train_image_names
            for caption in captions_by_image[image_name]
        )

        encoded_captions = {}
        maximum_caption_length = 0
        for image_name, captions in captions_by_image.items():
            caption_pairs = []
            for caption in captions:
                input_ids, target_ids = tokenizer.encode_pair(caption)
                input_tensor = torch.tensor(input_ids, dtype=torch.long)    # [L+1]
                target_tensor = torch.tensor(target_ids, dtype=torch.long)  # [L+1]
                maximum_caption_length = max(
                    maximum_caption_length,
                    input_tensor.numel(),
                )
                caption_pairs.append((input_tensor, target_tensor))
            encoded_captions[image_name] = caption_pairs

        tokenizer.save(self.tokenizer_path)
        return tokenizer, encoded_captions, maximum_caption_length

    def _load_feature_bank(self, image_names):
        # Input: saved cache [N_images,512,7,7].
        # Output: FP16 feature bank plus dict[image_name, row_index].
        if not self.feature_cache_path.exists():
            raise FileNotFoundError(
                f"Missing {self.feature_cache_path}. "
                "Run build_image_feature_cache.py first."
            )

        feature_cache = torch.load(
            self.feature_cache_path,
            map_location="cpu",
            weights_only=True,
        )
        feature_bank = feature_cache["features"].contiguous()       # [N,512,7,7]
        feature_index = {
            image_name: index
            for index, image_name in enumerate(feature_cache["image_names"])
        }

        missing_images = set(image_names) - set(feature_index)
        if missing_images:
            raise ValueError(
                f"Feature cache is missing {len(missing_images)} caption images"
            )
        if feature_bank.ndim != 4 or tuple(feature_bank.shape[1:]) != (512, 7, 7):
            raise ValueError(
                f"Expected feature bank [N,512,7,7], got {tuple(feature_bank.shape)}"
            )
        return feature_bank, feature_index

    def _make_loader(self, dataset, collator, shuffle):
        # Input: grouped dataset items.
        # Output per batch: [B_img,512,7,7], [B_cap,T], [B_cap,T], [B_cap].
        return DataLoader(
            dataset,
            batch_size=self.image_batch_size,
            shuffle=shuffle,
            collate_fn=collator,
            num_workers=0,
            # The 406 MB bank is already in RAM. On Windows, worker processes
            # would copy it and usually make this cached loader slower.
            pin_memory=True,
            drop_last=False,
        )

    def run(self):
        # Input: configured CSV and feature-cache paths.
        # Output: fully constructed CaptionDataBundle.
        captions_by_image = self._read_captions()
        all_image_names = list(captions_by_image)
        train_names, val_names, test_names = self._split_images(all_image_names)

        tokenizer, encoded_captions, max_length = self._fit_and_encode(
            captions_by_image,
            train_names,
        )
        feature_bank, feature_index = self._load_feature_bank(all_image_names)

        train_dataset = GroupedCachedCaptionDataset(
            train_names, feature_bank, feature_index, encoded_captions
        )
        val_dataset = GroupedCachedCaptionDataset(
            val_names, feature_bank, feature_index, encoded_captions
        )
        test_dataset = GroupedCachedCaptionDataset(
            test_names, feature_bank, feature_index, encoded_captions
        )

        collator = GroupedCaptionCollator(tokenizer.pad_id)
        train_loader = self._make_loader(train_dataset, collator, shuffle=True)
        val_loader = self._make_loader(val_dataset, collator, shuffle=False)
        test_loader = self._make_loader(test_dataset, collator, shuffle=False)

        return CaptionDataBundle(
            tokenizer=tokenizer,
            feature_bank=feature_bank,
            feature_index=feature_index,
            encoded_captions=encoded_captions,
            train_image_names=train_names,
            val_image_names=val_names,
            test_image_names=test_names,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            test_dataset=test_dataset,
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            maximum_caption_length=max_length,
        )


## 6. Run the data pipeline


In [44]:
# PURPOSE: Execute the complete data-preparation pipeline through one public call.
# INPUT SHAPES: Caption CSV [N_captions], feature cache [N_images,512,7,7].
# OUTPUT SHAPE: data bundle with loaders yielding grouped caption batches.

data_pipeline = CaptionDataPipeline(
    caption_file=CAPTION_FILE,
    feature_cache_path=FEATURE_CACHE_PATH,
    tokenizer_path=TOKENIZER_PATH,
    train_size=TRAIN_SIZE,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    max_vocab_size=MAX_VOCAB_SIZE,
    min_token_frequency=MIN_TOKEN_FREQUENCY,
    image_batch_size=IMAGE_BATCH_SIZE,
    seed=SEED,
)
data = data_pipeline.run()

# Short aliases keep the model and training cells readable.
tokenizer = data.tokenizer
feature_bank = data.feature_bank                              # [N_images,512,7,7]
feature_index = data.feature_index
encoded_captions = data.encoded_captions
train_dataset = data.train_dataset
val_dataset = data.val_dataset
test_dataset = data.test_dataset
train_loader = data.train_loader
val_loader = data.val_loader
test_loader = data.test_loader
maximum_caption_length = data.maximum_caption_length

print("Train/validation/test images:", len(data.train_image_names), len(data.val_image_names), len(data.test_image_names))
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum caption length:", maximum_caption_length)
print("Feature bank:", feature_bank.shape, feature_bank.dtype)


Train/validation/test images: 5662 810 1619
Vocabulary size: 4385
Maximum caption length: 37
Feature bank: torch.Size([8091, 512, 7, 7]) torch.float16


## 7. Batch shape check


In [45]:
# PURPOSE: Verify the pipeline's final batch contract before model construction.
# INPUT SHAPE: B_img grouped image records from train_dataset.
# OUTPUT SHAPES: features [B_img,512,7,7], captions [B_cap,T], owners [B_cap].

feature_maps, caption_inputs, caption_targets, caption_image_indices = next(
    iter(train_loader)
)

print("Feature maps:", feature_maps.shape)
print("Caption inputs:", caption_inputs.shape)
print("Caption targets:", caption_targets.shape)
print("Caption image indices:", caption_image_indices.shape)


Feature maps: torch.Size([8, 512, 7, 7])
Caption inputs: torch.Size([40, 22])
Caption targets: torch.Size([40, 22])
Caption image indices: torch.Size([40])


## 8. Vectorized sinusoidal positional encoding


In [46]:
# PURPOSE: Precompute positional values once instead of using Python/GPU loops.
# INPUT SHAPE: Caption embeddings [B_cap, T, E].
# OUTPUT SHAPE: Position-aware embeddings [B_cap, T, E].

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_length):
        super().__init__()
        positions = torch.arange(max_length, dtype=torch.float32).unsqueeze(1)  # [Tmax,1]
        even_dimensions = torch.arange(0, embed_dim, 2, dtype=torch.float32)    # [E/2]
        frequencies = torch.exp(
            even_dimensions * (-math.log(10_000.0) / embed_dim)
        )                                                                       # [E/2]

        encoding = torch.zeros(max_length, embed_dim, dtype=torch.float32)      # [Tmax,E]
        encoding[:, 0::2] = torch.sin(positions * frequencies)                  # [Tmax,E/2]
        encoding[:, 1::2] = torch.cos(positions * frequencies)                  # [Tmax,E/2]
        self.register_buffer("encoding", encoding.unsqueeze(0), persistent=False)  # [1,Tmax,E]

    def forward(self, embeddings):
        # embeddings: [B_cap,T,E] -> output: [B_cap,T,E]
        caption_length = embeddings.size(1)
        if caption_length > self.encoding.size(1):
            raise ValueError("Caption is longer than the positional-encoding buffer")
        return embeddings + self.encoding[:, :caption_length].to(embeddings.dtype)


## 9. Cached feature projector


In [47]:
# PURPOSE: Project frozen ResNet channels and reuse each image for its captions.
# INPUT SHAPES: features [B_img,512,7,7], caption_image_indices [B_cap].
# OUTPUT SHAPE: caption-specific image memory [B_cap,49,E].

class CachedFeatureProjector(nn.Module):
    def __init__(self, input_channels, embed_dim):
        super().__init__()
        self.projection = nn.Conv2d(input_channels, embed_dim, kernel_size=1)
        self.normalization = nn.LayerNorm(embed_dim)

    def forward(self, feature_maps, caption_image_indices):
        projected = self.projection(feature_maps)                 # [B_img,E,7,7]
        projected = projected.flatten(2).transpose(1, 2)          # [B_img,49,E]
        projected = self.normalization(projected)                 # [B_img,49,E]
        return projected.index_select(0, caption_image_indices)   # [B_cap,49,E]


## 10. Handwritten masked multi-head self-attention


In [48]:
# PURPOSE: Apply your handwritten causal self-attention with padding masking.
# INPUT SHAPES: input_batch [B_cap,T,E], key_padding_mask [B_cap,T].
# OUTPUT SHAPE: attended caption states [B_cap,T,E].

class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, masked=False):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"Cannot divide embed_dim={embed_dim} into {num_heads} heads"
            )

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.masked = masked

        self.q = nn.Linear(embed_dim, embed_dim)
        self.k = nn.Linear(embed_dim, embed_dim)
        self.v = nn.Linear(embed_dim, embed_dim)
        self.Wo = nn.Linear(embed_dim, embed_dim)

    def forward(self, input_batch, key_padding_mask=None):
        batch_size, sequence_length, embed_dim = input_batch.shape  # [B_cap,T,E]
        heads = self.num_heads
        head_dim = self.head_dim

        query = self.q(input_batch)                                # [B_cap,T,E]
        key = self.k(input_batch)                                  # [B_cap,T,E]
        value = self.v(input_batch)                                # [B_cap,T,E]

        query = query.reshape(
            batch_size, sequence_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,T,Eh]
        key = key.reshape(
            batch_size, sequence_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,T,Eh]
        value = value.reshape(
            batch_size, sequence_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,T,Eh]

        similarity = query @ key.transpose(-2, -1)                 # [B_cap,H,T,T]
        similarity = similarity / math.sqrt(head_dim)              # [B_cap,H,T,T]

        if self.masked:
            causal_mask = torch.triu(
                torch.ones(
                    sequence_length,
                    sequence_length,
                    dtype=torch.bool,
                    device=input_batch.device,
                ),
                diagonal=1,
            )                                                      # [T,T]
            similarity = similarity.masked_fill(
                causal_mask.view(1, 1, sequence_length, sequence_length),
                float("-inf"),
            )                                                      # [B_cap,H,T,T]

        if key_padding_mask is not None:
            similarity = similarity.masked_fill(
                key_padding_mask[:, None, None, :],                # [B_cap,1,1,T]
                float("-inf"),
            )                                                      # [B_cap,H,T,T]

        attention_weights = torch.softmax(similarity, dim=-1)      # [B_cap,H,T,T]
        attended = attention_weights @ value                       # [B_cap,H,T,Eh]
        attended = attended.transpose(1, 2).contiguous().reshape(
            batch_size, sequence_length, embed_dim
        )                                                          # [B_cap,T,E]
        return self.Wo(attended)                                   # [B_cap,T,E]


## 11. Handwritten image cross-attention


In [49]:
# PURPOSE: Attend from caption states to cached spatial image positions.
# INPUT SHAPES: decoder states [B_cap,T,E], image memory [B_cap,49,E].
# OUTPUT SHAPE: image-conditioned caption states [B_cap,T,E].

class CrossAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        if embed_dim % num_heads != 0:
            raise ValueError(
                f"Cannot divide embed_dim={embed_dim} into {num_heads} heads"
            )

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.q_w = nn.Linear(embed_dim, embed_dim)
        self.k_w = nn.Linear(embed_dim, embed_dim)
        self.v_w = nn.Linear(embed_dim, embed_dim)
        self.o_w = nn.Linear(embed_dim, embed_dim)

    def forward(self, batch_dec, batch_enc):
        batch_size, decoder_length, embed_dim = batch_dec.shape    # [B_cap,T,E]
        _, encoder_length, _ = batch_enc.shape                     # [B_cap,49,E]
        heads = self.num_heads
        head_dim = self.head_dim

        query = self.q_w(batch_dec)                                # [B_cap,T,E]
        key = self.k_w(batch_enc)                                  # [B_cap,49,E]
        value = self.v_w(batch_enc)                                # [B_cap,49,E]

        query = query.reshape(
            batch_size, decoder_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,T,Eh]
        key = key.reshape(
            batch_size, encoder_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,49,Eh]
        value = value.reshape(
            batch_size, encoder_length, heads, head_dim
        ).transpose(1, 2)                                          # [B_cap,H,49,Eh]

        similarity = query @ key.transpose(-2, -1)                 # [B_cap,H,T,49]
        similarity = similarity / math.sqrt(head_dim)              # [B_cap,H,T,49]
        attention_weights = torch.softmax(similarity, dim=-1)      # [B_cap,H,T,49]
        attended = attention_weights @ value                       # [B_cap,H,T,Eh]
        attended = attended.transpose(1, 2).contiguous().reshape(
            batch_size, decoder_length, embed_dim
        )                                                          # [B_cap,T,E]
        return self.o_w(attended)                                  # [B_cap,T,E]


## 12. Handwritten decoder block


In [50]:
# PURPOSE: Combine masked self-attention, image cross-attention, and an FFN.
# INPUT SHAPES: caption states [B_cap,T,E], image memory [B_cap,49,E], mask [B_cap,T].
# OUTPUT SHAPE: transformed caption states [B_cap,T,E].

class DecoderBlockCrossAttention(nn.Module):
    def __init__(self, embed_dim, attention_heads, dropout=0.1):
        super().__init__()
        self.MaskedAttention = MultiHeadAttention(
            embed_dim,
            attention_heads,
            masked=True,
        )
        self.CrossAttention = CrossAttention(embed_dim, attention_heads)
        self.fnn = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(4 * embed_dim, embed_dim),
        )
        self.dropout = nn.Dropout(dropout)
        self.layerNorm1 = nn.LayerNorm(embed_dim)
        self.layerNorm2 = nn.LayerNorm(embed_dim)
        self.layerNorm3 = nn.LayerNorm(embed_dim)

    def forward(self, input_embeds, batch_enc, key_padding_mask=None):
        self_attention = self.MaskedAttention(
            input_embeds,
            key_padding_mask=key_padding_mask,
        )                                                          # [B_cap,T,E]
        output = self.layerNorm1(
            input_embeds + self.dropout(self_attention)
        )                                                          # [B_cap,T,E]

        cross_attention = self.CrossAttention(output, batch_enc)   # [B_cap,T,E]
        output = self.layerNorm2(
            output + self.dropout(cross_attention)
        )                                                          # [B_cap,T,E]

        feed_forward = self.fnn(output)                            # [B_cap,T,E]
        return self.layerNorm3(
            output + self.dropout(feed_forward)
        )                                                          # [B_cap,T,E]


## 13. Full handwritten decoder with cached image features


In [51]:
# PURPOSE: Stack your custom decoder blocks over cached spatial image features.
# INPUT SHAPES: features [B_img,512,7,7], token IDs [B_cap,T], owners [B_cap].
# OUTPUT SHAPE: vocabulary logits [B_cap,T,V].

class DecoderWithCrossAttention(nn.Module):
    def __init__(
        self,
        vocab_count,
        pad_id,
        max_caption_length,
        embedding_dim=256,
        attention_heads=8,
        n_blocks=2,
        dropout=0.1,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.embedding_scale = math.sqrt(embedding_dim)
        self.feature_projector = CachedFeatureProjector(512, embedding_dim)
        self.embedding = nn.Embedding(
            vocab_count,
            embedding_dim,
            padding_idx=pad_id,
        )
        self.add_positional = SinusoidalPositionalEncoding(
            embedding_dim,
            max_caption_length,
        )
        self.decoders = nn.ModuleList(
            DecoderBlockCrossAttention(
                embedding_dim,
                attention_heads,
                dropout=dropout,
            )
            for _ in range(n_blocks)
        )
        self.output_layer = nn.Linear(embedding_dim, vocab_count, bias=False)
        self.output_layer.weight = self.embedding.weight

    def forward(self, image_features, caption_input, caption_image_indices):
        image_memory = self.feature_projector(
            image_features,
            caption_image_indices,
        )                                                          # [B_cap,49,E]

        caption_states = (
            self.embedding(caption_input) * self.embedding_scale
        )                                                          # [B_cap,T,E]
        caption_states = self.add_positional(caption_states)       # [B_cap,T,E]
        padding_mask = caption_input.eq(self.pad_id)                # [B_cap,T]

        for block in self.decoders:
            caption_states = block(
                caption_states,
                image_memory,
                key_padding_mask=padding_mask,
            )                                                      # [B_cap,T,E]

        return self.output_layer(caption_states)                    # [B_cap,T,V]


## 14. Model, optimizer, AMP, and loss


In [52]:
# PURPOSE: Place the full trainable model on CUDA and configure optimization.
# INPUT SHAPE: Model expects [B_img,512,7,7], [B_cap,T], and [B_cap].
# OUTPUT SHAPE: Model returns logits [B_cap,T,V].

model = DecoderWithCrossAttention(
    vocab_count=tokenizer.vocab_size,
    pad_id=tokenizer.pad_id,
    max_caption_length=maximum_caption_length,
    embedding_dim=EMBED_DIM,
    attention_heads=ATTENTION_HEADS,
    n_blocks=DECODER_LAYERS,
    dropout=DROPOUT,
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fused=True,
)
criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_id)
scaler = torch.amp.GradScaler("cuda")

trainable_parameters = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)
print(f"Trainable parameters: {trainable_parameters:,}")


Trainable parameters: 3,361,280


## 15. One GPU shape check


In [53]:
# PURPOSE: Verify one cached batch passes through the CUDA model correctly.
# INPUT SHAPES: [B_img,512,7,7], [B_cap,T], [B_cap].
# OUTPUT SHAPE: logits [B_cap,T,V].

feature_maps, caption_inputs, caption_targets, caption_image_indices = next(
    iter(train_loader)
)
feature_maps = feature_maps.to(device, non_blocking=True)                    # [B_img,512,7,7]
caption_inputs = caption_inputs.to(device, non_blocking=True)                # [B_cap,T]
caption_image_indices = caption_image_indices.to(device, non_blocking=True)  # [B_cap]

model.eval()
with torch.inference_mode(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16,
):
    logits = model(
        feature_maps,
        caption_inputs,
        caption_image_indices,
    )                                                                        # [B_cap,T,V]

print("Logits:", logits.shape)
model.train()


Logits: torch.Size([40, 15, 4385])


DecoderWithCrossAttention(
  (feature_projector): CachedFeatureProjector(
    (projection): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (normalization): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  )
  (embedding): Embedding(4385, 256, padding_idx=0)
  (add_positional): SinusoidalPositionalEncoding()
  (decoders): ModuleList(
    (0-1): 2 x DecoderBlockCrossAttention(
      (MaskedAttention): MultiHeadAttention(
        (q): Linear(in_features=256, out_features=256, bias=True)
        (k): Linear(in_features=256, out_features=256, bias=True)
        (v): Linear(in_features=256, out_features=256, bias=True)
        (Wo): Linear(in_features=256, out_features=256, bias=True)
      )
      (CrossAttention): CrossAttention(
        (q_w): Linear(in_features=256, out_features=256, bias=True)
        (k_w): Linear(in_features=256, out_features=256, bias=True)
        (v_w): Linear(in_features=256, out_features=256, bias=True)
        (o_w): Linear(in_features=256, out_f

## 16. GPU mixed-precision training loop


In [54]:
# PURPOSE: Train only the cached-feature projector and Transformer decoder.
# INPUT SHAPES PER BATCH: features [B_img,512,7,7], captions [B_cap,T], owners [B_cap].
# OUTPUT SHAPE PER BATCH: logits [B_cap,T,V]; scalar cross-entropy loss.

model.train()

for epoch in range(EPOCHS):
    epoch_loss = 0.0

    for (
        feature_maps,
        caption_inputs,
        caption_targets,
        caption_image_indices,
    ) in train_loader:
        feature_maps = feature_maps.to(device, non_blocking=True)                    # [B_img,512,7,7]
        caption_inputs = caption_inputs.to(device, non_blocking=True)                # [B_cap,T]
        caption_targets = caption_targets.to(device, non_blocking=True)              # [B_cap,T]
        caption_image_indices = caption_image_indices.to(device, non_blocking=True)  # [B_cap]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(
                feature_maps,
                caption_inputs,
                caption_image_indices,
            )                                                                        # [B_cap,T,V]
            loss = criterion(
                logits.reshape(-1, tokenizer.vocab_size),                            # [B_cap*T,V]
                caption_targets.reshape(-1),                                         # [B_cap*T]
            )                                                                        # scalar

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    average_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch + 1:03d}/{EPOCHS} | train loss: {average_loss:.4f}")


Epoch 001/100 | train loss: 19.9952
Epoch 002/100 | train loss: 8.8106
Epoch 003/100 | train loss: 6.9836
Epoch 004/100 | train loss: 6.0657
Epoch 005/100 | train loss: 5.4754
Epoch 006/100 | train loss: 5.0514
Epoch 007/100 | train loss: 4.7331
Epoch 008/100 | train loss: 4.4997
Epoch 009/100 | train loss: 4.3116
Epoch 010/100 | train loss: 4.1502
Epoch 011/100 | train loss: 4.0110
Epoch 012/100 | train loss: 3.8943
Epoch 013/100 | train loss: 3.7826
Epoch 014/100 | train loss: 3.6833
Epoch 015/100 | train loss: 3.5927
Epoch 016/100 | train loss: 3.5127
Epoch 017/100 | train loss: 3.4347
Epoch 018/100 | train loss: 3.3684
Epoch 019/100 | train loss: 3.3058
Epoch 020/100 | train loss: 3.2488
Epoch 021/100 | train loss: 3.1935
Epoch 022/100 | train loss: 3.1391
Epoch 023/100 | train loss: 3.0945
Epoch 024/100 | train loss: 3.0451
Epoch 025/100 | train loss: 2.9971
Epoch 026/100 | train loss: 2.9514
Epoch 027/100 | train loss: 2.9075
Epoch 028/100 | train loss: 2.8643
Epoch 029/100 | tra

## 17. Validation


In [55]:
# PURPOSE: Measure validation loss without gradients.
# INPUT SHAPES: features [B_img,512,7,7], captions [B_cap,T], owners [B_cap].
# OUTPUT SHAPE: One scalar token-weighted validation loss.

def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.inference_mode():
        for features, inputs, targets, owners in data_loader:
            features = features.to(device, non_blocking=True)  # [B_img,512,7,7]
            inputs = inputs.to(device, non_blocking=True)      # [B_cap,T]
            targets = targets.to(device, non_blocking=True)    # [B_cap,T]
            owners = owners.to(device, non_blocking=True)      # [B_cap]

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(features, inputs, owners)       # [B_cap,T,V]
                token_losses = nn.functional.cross_entropy(
                    logits.reshape(-1, tokenizer.vocab_size),  # [B_cap*T,V]
                    targets.reshape(-1),                       # [B_cap*T]
                    ignore_index=tokenizer.pad_id,
                    reduction="sum",
                )                                              # scalar summed loss

            valid_tokens = targets.ne(tokenizer.pad_id).sum().item()
            total_loss += token_losses.item()
            total_tokens += valid_tokens

    model.train()
    return total_loss / total_tokens


validation_loss = evaluate(model, val_loader)
print(f"Validation loss: {validation_loss:.4f}")


Validation loss: 4.9256


## 18. Save decoder checkpoint


In [56]:
# PURPOSE: Save everything required to restore the trained decoder.
# INPUT SHAPE: Model/optimizer states and scalar metadata.
# OUTPUT SHAPE: One checkpoint file on disk.

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
checkpoint_path = CHECKPOINT_DIR / "caption_decoder_pipeline.pt"
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "vocab_size": tokenizer.vocab_size,
        "pad_id": tokenizer.pad_id,
        "max_caption_length": maximum_caption_length,
        "embed_dim": EMBED_DIM,
        "attention_heads": ATTENTION_HEADS,
        "decoder_layers": DECODER_LAYERS,
        "feature_cache": str(FEATURE_CACHE_PATH),
        "tokenizer": str(TOKENIZER_PATH),
    },
    checkpoint_path,
)
print("Saved checkpoint:", checkpoint_path)


Saved checkpoint: checkpoints\caption_decoder_pipeline.pt
